In [1]:
import os
import csv
import numpy as np
import pandas as pd
import mediapipe as mp
import cv2
from enum import Enum


class BodyPart(Enum):
    """Body part enum with standard pose landmark indices used in MediaPipe Pose."""
    NOSE = 0
    LEFT_EYE_INNER = 1
    LEFT_EYE = 2
    LEFT_EYE_OUTER = 3
    RIGHT_EYE_INNER = 4
    RIGHT_EYE = 5
    RIGHT_EYE_OUTER = 6
    LEFT_EAR = 7
    RIGHT_EAR = 8
    MOUTH_LEFT = 9
    MOUTH_RIGHT = 10
    LEFT_SHOULDER = 11
    RIGHT_SHOULDER = 12
    LEFT_ELBOW = 13
    RIGHT_ELBOW = 14
    LEFT_WRIST = 15
    RIGHT_WRIST = 16
    LEFT_PINKY = 17
    RIGHT_PINKY = 18
    LEFT_INDEX = 19
    RIGHT_INDEX = 20
    LEFT_THUMB = 21
    RIGHT_THUMB = 22
    LEFT_HIP = 23
    RIGHT_HIP = 24
    LEFT_KNEE = 25
    RIGHT_KNEE = 26
    LEFT_ANKLE = 27
    RIGHT_ANKLE = 28
    LEFT_HEEL = 29
    RIGHT_HEEL = 30
    LEFT_FOOT_INDEX = 31
    RIGHT_FOOT_INDEX = 32


class Person:
    """Person data structure with pose landmarks."""

    def __init__(self, keypoints):
        self.keypoints = keypoints


class Preprocessor:
    def __init__(self, images_in_folder, csvs_out_path):
        self._images_in_folder = images_in_folder
        self._csvs_out_path = csvs_out_path
        self._csvs_out_folder_per_class = 'csv_per_pose'
        self._messages = []

        # Initialize MediaPipe
        self.mp_pose = mp.solutions.pose
        self.pose = self.mp_pose.Pose(
            static_image_mode=True,
            model_complexity=2,  # 2 is the highest accuracy model
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5
        )

        os.makedirs(self._csvs_out_folder_per_class, exist_ok=True)

        # Find all pose class folders
        self._pose_class_names = sorted([
            n for n in os.listdir(self._images_in_folder)
            if os.path.isdir(os.path.join(self._images_in_folder, n))
        ])

    def detect_pose(self, image):
        """Detect pose landmarks using MediaPipe."""
        # Convert the image to RGB as MediaPipe requires RGB input
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Process the image
        results = self.pose.process(image_rgb)

        if not results.pose_landmarks:
            return None

        # Convert MediaPipe landmarks to our custom format
        keypoints = []
        for idx, landmark in enumerate(results.pose_landmarks.landmark):
            keypoints.append({
                'coordinate': {'x': landmark.x, 'y': landmark.y},
                'score': landmark.visibility,
                'part': idx
            })

        return Person(keypoints)

    def process(self, detection_threshold=0.1):
        """Process all images and extract pose landmarks."""
        for pose_class_name in self._pose_class_names:
            print(f"Preprocessing {pose_class_name}...")

            images_in_folder = os.path.join(self._images_in_folder, pose_class_name)
            csv_out_path = os.path.join(self._csvs_out_folder_per_class, f"{pose_class_name}.csv")

            with open(csv_out_path, 'w', newline='') as csv_out_file:
                csv_out_writer = csv.writer(csv_out_file, delimiter=',', quoting=csv.QUOTE_MINIMAL)

                image_names = sorted([
                    n for n in os.listdir(images_in_folder)
                    if not n.startswith('.')
                ])

                valid_image_count = 0

                for index, image_name in enumerate(image_names):
                    print(f"Processing {index + 1}/{len(image_names)}: {image_name}")
                    image_path = os.path.join(images_in_folder, image_name)

                    try:
                        image = cv2.imread(image_path)
                        if image is None:
                            self._messages.append(f"Skipped {image_path}: Could not read image")
                            continue
                    except Exception as e:
                        self._messages.append(f"Skipped {image_path}: Invalid image: {str(e)}")
                        continue

                    if len(image.shape) != 3 or image.shape[2] != 3:
                        self._messages.append(f"Skipped {image_path}: Image is not in RGB format")
                        continue

                    person = self.detect_pose(image)

                    if person is None:
                        self._messages.append(f"Skipped {image_path}: No person detected")
                        continue

                    # Check if all keypoints meet the detection threshold
                    min_landmark_score = min([keypoint['score'] for keypoint in person.keypoints])
                    if min_landmark_score < detection_threshold:
                        self._messages.append(f"Skipped {image_path}: Keypoint score below threshold")
                        continue

                    valid_image_count += 1

                    # Extract landmarks for CSV
                    pose_landmarks = np.array([
                        [keypoint['coordinate']['x'], keypoint['coordinate']['y'], keypoint['score']]
                        for keypoint in person.keypoints
                    ], dtype=np.float32)

                    coord = pose_landmarks.flatten().astype(str).tolist()
                    csv_out_writer.writerow([image_name] + coord)

            print(f"Successfully processed {valid_image_count}/{len(image_names)} images for '{pose_class_name}'")

        # Combine all landmarks into a single dataframe
        all_landmarks_df = self._all_landmarks_as_dataframe()
        if all_landmarks_df.empty:
            print("Warning: No valid landmarks found. Skipping final CSV creation.")
        else:
            all_landmarks_df.to_csv(self._csvs_out_path, index=False)
            print(f"Combined CSV file saved at: {self._csvs_out_path}")

        # Print any warning messages
        if self._messages:
            print("Messages/Warnings during preprocessing:")
            for message in self._messages:
                print(message)

    def class_names(self):
        """Return list of pose class names."""
        return self._pose_class_names

    def _all_landmarks_as_dataframe(self):
        """Combine all per-class CSVs into a single dataframe."""
        total_df = None
        for class_index, class_name in enumerate(self._pose_class_names):
            csv_out_path = os.path.join(self._csvs_out_folder_per_class, f"{class_name}.csv")

            # Check if the CSV file exists and is not empty
            if not os.path.exists(csv_out_path) or os.path.getsize(csv_out_path) == 0:
                print(f"Warning: CSV file '{csv_out_path}' is empty or does not exist. Skipping.")
                continue

            per_class_df = pd.read_csv(csv_out_path, header=None)

            per_class_df['class_no'] = [class_index] * len(per_class_df)
            per_class_df['class_name'] = [class_name] * len(per_class_df)
            per_class_df[per_class_df.columns[0]] = (
                    class_name + '/' + per_class_df[per_class_df.columns[0]])

            if total_df is None:
                total_df = per_class_df
            else:
                total_df = pd.concat([total_df, per_class_df], axis=0)

        if total_df is None:
            print("Warning: No valid CSV files found. Returning an empty DataFrame.")
            return pd.DataFrame()

        # Create column names based on body parts
        list_name = []
        for i in range(33):  # MediaPipe provides 33 landmarks
            # Get the name of the body part from the enum
            bodypart_name = next((bp.name for bp in BodyPart if bp.value == i), f"UNKNOWN_{i}")
            list_name.append([f"{bodypart_name}_x", f"{bodypart_name}_y", f"{bodypart_name}_score"])

        header_name = ['filename']
        for columns_name in list_name:
            header_name += columns_name
        header_name += ['class_no', 'class_name']

        # Map column indices to named columns
        header_map = {total_df.columns[i]: header_name[i] for i in range(len(header_name))}
        total_df.rename(header_map, axis=1, inplace=True)

        return total_df

    def __del__(self):
        """Clean up MediaPipe resources."""
        if hasattr(self, 'pose'):
            self.pose.close()

In [2]:
import os
import time
from preprocessor import Preprocessor

def main():
    """Main function to process images and generate CSV files with pose landmarks."""
    print("Starting pose landmark extraction with MediaPipe...")
    
    # Paths configuration
    train_images_folder = 'Final_datasets/Train'
    train_csv_output = 'train_data_final.csv'
    
    # Optional: Also process validation data
    val_images_folder = 'Final_datasets/Validation'
    val_csv_output = 'val_data_final.csv'
    
    # Added test data configuration
    test_images_folder = 'Final_datasets/Test'
    test_csv_output = 'test_data_final.csv'
    
    # Ensure the image folders exist
    if not os.path.exists(train_images_folder):
        print(f"Error: Training data folder '{train_images_folder}' not found.")
        return
    
    # Process training data
    print(f"\nProcessing training data from: {train_images_folder}")
    start_time = time.time()
    
    train_preprocessor = Preprocessor(train_images_folder, train_csv_output)
    train_preprocessor.process(detection_threshold=0.1)
    
    print(f"Training data processing completed in {time.time() - start_time:.2f} seconds")
    print(f"Pose classes found: {train_preprocessor.class_names()}")
    
    # Process validation data if the folder exists
    if os.path.exists(val_images_folder):
        print(f"\nProcessing validation data from: {val_images_folder}")
        start_time = time.time()
        
        val_preprocessor = Preprocessor(val_images_folder, val_csv_output)
        val_preprocessor.process(detection_threshold=0.1)
        
        print(f"Validation data processing completed in {time.time() - start_time:.2f} seconds")
    
    # Process test data if the folder exists
    if os.path.exists(test_images_folder):
        print(f"\nProcessing test data from: {test_images_folder}")
        start_time = time.time()
        
        test_preprocessor = Preprocessor(test_images_folder, test_csv_output)
        test_preprocessor.process(detection_threshold=0.1)
        
        print(f"Test data processing completed in {time.time() - start_time:.2f} seconds")
    
    print("\nAll processing completed successfully!")

if __name__ == "__main__":
    main()

Starting pose landmark extraction with MediaPipe...

Processing training data from: Final_datasets/Train
Preprocessing chair...
Processing 1/400: girl1_chair070.jpg
Processing 2/400: girl1_chair070_flipped.jpg
Processing 3/400: girl1_chair075.jpg
Processing 4/400: girl1_chair075_flipped.jpg
Processing 5/400: girl1_chair076.jpg
Processing 6/400: girl1_chair076_flipped.jpg
Processing 7/400: girl1_chair080.jpg
Processing 8/400: girl1_chair080_flipped.jpg
Processing 9/400: girl1_chair081.jpg
Processing 10/400: girl1_chair081_flipped.jpg
Processing 11/400: girl1_chair085.jpg
Processing 12/400: girl1_chair085_flipped.jpg
Processing 13/400: girl1_chair090.jpg
Processing 14/400: girl1_chair090_flipped.jpg
Processing 15/400: girl1_chair091.jpg
Processing 16/400: girl1_chair091_flipped.jpg
Processing 17/400: girl1_chair094.jpg
Processing 18/400: girl1_chair094_flipped.jpg
Processing 19/400: girl1_chair104.jpg
Processing 20/400: girl1_chair104_flipped.jpg
Processing 21/400: girl1_chair106.jpg
Pro

In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

class PoseClassifier:
    """A classifier for pose landmarks extracted with MediaPipe Pose."""
    
    def __init__(self, model_path=None):
        """Initialize the classifier."""
        if model_path:
            self.pipeline = joblib.load(model_path)
        else:
            self.pipeline = Pipeline([
                ('scaler', StandardScaler()),
                ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
            ])
        
    def load_data(self, csv_path):
        """Load pose landmark data from a CSV file."""
        df = pd.read_csv(csv_path)
        
        # The first column is the filename, the last two are class_no and class_name
        # Everything in between is landmark data (x, y, score)
        X = df.iloc[:, 1:-2].values
        y = df['class_name'].values
        
        return X, y
        
    def train(self, train_csv_path, validation_csv_path=None, optimize=False):
        """Train the pose classifier model."""
        print("Loading training data...")
        X_train, y_train = self.load_data(train_csv_path)
        
        if validation_csv_path:
            print("Loading validation data...")
            X_val, y_val = self.load_data(validation_csv_path)
        else:
            print("Splitting training data for validation...")
            X_train, X_val, y_train, y_val = train_test_split(
                X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
            )
        
        if optimize:
            print("Optimizing model hyperparameters...")
            param_grid = {
                'classifier__n_estimators': [50, 100, 200],
                'classifier__max_depth': [None, 10, 20, 30],
                'classifier__min_samples_split': [2, 5, 10],
            }
            
            grid_search = GridSearchCV(
                self.pipeline, param_grid, cv=5, n_jobs=-1, verbose=1
            )
            grid_search.fit(X_train, y_train)
            
            print(f"Best parameters: {grid_search.best_params_}")
            self.pipeline = grid_search.best_estimator_
        else:
            print("Training model with default parameters...")
            self.pipeline.fit(X_train, y_train)
        
        # Evaluate the model
        y_pred = self.pipeline.predict(X_val)
        accuracy = accuracy_score(y_val, y_pred)
        print(f"Validation accuracy: {accuracy:.4f}")
        
        print("\nClassification Report:")
        print(classification_report(y_val, y_pred))
        
        return accuracy
        
    def predict(self, landmarks):
        """Predict pose class from landmarks."""
        # Ensure the landmarks are in the right shape
        landmarks = np.array(landmarks).reshape(1, -1)
        
        # Predict using the trained pipeline
        prediction = self.pipeline.predict(landmarks)[0]
        confidence = np.max(self.pipeline.predict_proba(landmarks))
        
        return prediction, confidence
        
    def save_model(self, model_path):
        """Save the trained model to a file."""
        joblib.dump(self.pipeline, model_path)
        print(f"Model saved to {model_path}")
        
    def visualize_results(self, validation_csv_path, output_path="confusion_matrix.png"):
        """Visualize model results with a confusion matrix."""
        X_val, y_val = self.load_data(validation_csv_path)
        y_pred = self.pipeline.predict(X_val)
        
        # Create confusion matrix
        cm = confusion_matrix(y_val, y_pred)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=np.unique(y_val), 
                    yticklabels=np.unique(y_val))
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.title('Confusion Matrix')
        plt.tight_layout()
        plt.savefig(output_path)
        print(f"Confusion matrix saved to {output_path}")


In [4]:
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split

# 📌 Load the datasets
df_train = pd.read_csv('train_data_final.csv')
df_test = pd.read_csv('test_data_final.csv')

# Extract unique class names from the "class_name" column
class_names = df_train['class_name'].unique()
print("Unique class names:", class_names)

# 📌 Process training data
train_data = df_train.drop(['filename', 'class_name'], axis=1)  # Keep 'class_no' for labels
# Drop all _score columns (keeping only X & Y coordinates)
train_columns_to_drop = [col for col in train_data.columns if '_score' in col]
train_data = train_data.drop(columns=train_columns_to_drop, axis=1)

# 📌 Extract Features and Labels for training data
X_train_full = train_data.drop('class_no', axis=1).values  # Features
y_train_full = train_data['class_no'].values  # Labels

# 📌 Process test data (similar preprocessing as training data)
test_data = df_test.drop(['filename', 'class_name'], axis=1)  # Keep 'class_no' for labels
# Drop all _score columns (keeping only X & Y coordinates)
test_columns_to_drop = [col for col in test_data.columns if '_score' in col]
test_data = test_data.drop(columns=test_columns_to_drop, axis=1)

# 📌 Extract Features and Labels for test data
X_test = test_data.drop('class_no', axis=1).values  # Features
y_test = test_data['class_no'].values  # Labels

# 📌 One-hot encode labels
num_classes = len(set(y_train_full))  # Get the number of classes
y_train_full = tf.keras.utils.to_categorical(y_train_full, num_classes)
y_test = tf.keras.utils.to_categorical(y_test, num_classes)

# 📌 Split training data into train & validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.15, random_state=42)
print(X_train[10])  # Print sample data point

# 📌 Define Neural Network
tf.keras.backend.clear_session()  # Free memory

inputs = tf.keras.Input(shape=(66,))
layer = tf.keras.layers.Dense(128, activation=tf.nn.relu6)(inputs)
layer = tf.keras.layers.Dense(64, activation=tf.nn.relu6)(layer)
layer = tf.keras.layers.Dropout(0.2)(layer)
outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(layer)

model = tf.keras.Model(inputs, outputs)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

# 📌 Model Summary
model.summary()

# Set up callbacks
checkpoint_path = "weights.best.keras"
checkpoint = keras.callbacks.ModelCheckpoint(checkpoint_path,
                                           monitor='val_accuracy',
                                           verbose=1,
                                           save_best_only=True,
                                           mode='max')

earlystopping = keras.callbacks.EarlyStopping(monitor='val_accuracy',
                                            patience=20)

# Train the model
history = model.fit(X_train, y_train,
                    epochs=20,
                    batch_size=16,
                    validation_data=(X_val, y_val),
                    callbacks=[checkpoint, earlystopping])

# Load the best model weights
model.load_weights(checkpoint_path)

# Evaluate on both training and test datasets
train_loss, train_accuracy = model.evaluate(X_train, y_train)
print(f"Training accuracy: {train_accuracy*100:.2f}%")

test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_accuracy*100:.2f}%")

# Save the model
model.save("my_model_final.keras")
model.save("my_model_final.h5")

Unique class names: ['chair' 'cobra' 'downdog' 'plank' 'shoudler_stand' 'tree']
[0.5984605  0.32661077 0.60370064 0.316097   0.60510874 0.31667295
 0.60662216 0.31760496 0.59901714 0.31479132 0.59631836 0.3144404
 0.59280384 0.31424582 0.60475653 0.32470602 0.5895872  0.32066083
 0.5971679  0.34057674 0.5882166  0.34207776 0.61137503 0.3855462
 0.5637171  0.37165353 0.71161336 0.31324825 0.5518943  0.29852298
 0.71717954 0.21101317 0.69398224 0.2135013  0.72159165 0.18887815
 0.7191231  0.1902673  0.71553797 0.18420753 0.707562   0.18801552
 0.7102513  0.19292933 0.6837182  0.19755122 0.57570225 0.6195016
 0.5750748  0.6223393  0.6115101  0.7290921  0.62340117 0.721426
 0.59764683 0.8840157  0.60651344 0.8822696  0.58836037 0.91443664
 0.5953687  0.9083776  0.6288315  0.91643727 0.6337149  0.9208859 ]


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 6

C:\Users\Hello Tejas\AppData\Roaming\Python\Python39\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
